# Customer Data Engineering POC

Demonstrated a small Databricks/PySpark pipeline using Bronze/Silver/Gold,
data quality, deduplication, incremental processing and Delta MERGE.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

BASE_PATH = "/Volumes/databricks-customer-data-poc/data/raw_files"  
INITIAL_PATH = f"{BASE_PATH}/customers_initial.csv"
INCREMENTAL_PATH = f"{BASE_PATH}/customers_incremental.csv"

In [ ]:
df_source = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(INITIAL_PATH)
)

display(df_source)

CustomerID,CustomerName,Email,Country,UpdatedDate
101,John Smith,john@gmail.com,USA,2026-08-01
102,Pavan Kumar,pavan@gmail.com,India,2026-08-01
103,David Lee,david@gmail.com,UK,2026-08-02
104,Sarah Jones,sarah@gmail.com,USA,2026-08-02
105,Ravi Kumar,ravi@gmail.com,India,2026-08-03


## Bronze - raw Delta ingestion

In [ ]:
df_bronze = df_source.withColumn("ingestion_timestamp", F.current_timestamp())

df_bronze.write.format("delta").mode("overwrite").saveAsTable("bronze_customers")

display(spark.table("bronze_customers"))

CustomerID,CustomerName,Email,Country,UpdatedDate,ingestion_timestamp
101,John Smith,john@gmail.com,USA,2026-08-01,2026-08-19T06:49:16.114Z
102,Pavan Kumar,pavan@gmail.com,India,2026-08-01,2026-08-19T06:49:16.114Z
103,David Lee,david@gmail.com,UK,2026-08-02,2026-08-19T06:49:16.114Z
104,Sarah Jones,sarah@gmail.com,USA,2026-08-02,2026-08-19T06:49:16.114Z
105,Ravi Kumar,ravi@gmail.com,India,2026-08-03,2026-08-19T06:49:16.114Z


## Silver - cleansing and data quality

In [ ]:
df_silver = (
    spark.table("bronze_customers")
    .filter(F.col("CustomerID").isNotNull())
    .filter(F.col("Email").isNotNull())
    .withColumn("CustomerName", F.trim("CustomerName"))
    .withColumn("Email", F.lower(F.trim("Email")))
    .withColumn("Country", F.trim("Country"))
)

# Latest record per customer
window_spec = Window.partitionBy("CustomerID").orderBy(F.col("UpdatedDate").desc())

df_silver = (
    df_silver
    .withColumn("rn", F.row_number().over(window_spec))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

df_silver.write.format("delta").mode("overwrite").saveAsTable("silver_customers")

display(spark.table("silver_customers"))

CustomerID,CustomerName,Email,Country,UpdatedDate,ingestion_timestamp
101,John Smith,john@gmail.com,USA,2026-08-01,2026-08-19T06:49:16.114Z
102,Pavan Kumar,pavan@gmail.com,India,2026-08-01,2026-08-19T06:49:16.114Z
103,David Lee,david@gmail.com,UK,2026-08-02,2026-08-19T06:49:16.114Z
104,Sarah Jones,sarah@gmail.com,USA,2026-08-02,2026-08-19T06:49:16.114Z
105,Ravi Kumar,ravi@gmail.com,India,2026-08-03,2026-08-19T06:49:16.114Z


## Data quality metrics

In [ ]:
df = spark.table("silver_customers")

total_records = df.count()
null_customer_ids = df.filter(F.col("CustomerID").isNull()).count()
null_emails = df.filter(F.col("Email").isNull()).count()
duplicate_customers = (
    df.groupBy("CustomerID").count()
      .filter(F.col("count") > 1)
      .count()
)

print(f"Total Records       : {total_records}")
print(f"Null Customer IDs   : {null_customer_ids}")
print(f"Null Emails         : {null_emails}")
print(f"Duplicate Customers : {duplicate_customers}")

Total Records       : 5
Null Customer IDs   : 0
Null Emails         : 0
Duplicate Customers : 0


## incremental data

In [ ]:
df_incremental = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(INCREMENTAL_PATH)
)

df_incremental = (
    df_incremental
    .filter(F.col("CustomerID").isNotNull())
    .filter(F.col("Email").isNotNull())
    .withColumn("CustomerName", F.trim("CustomerName"))
    .withColumn("Email", F.lower(F.trim("Email")))
    .withColumn("Country", F.trim("Country"))
)

display(df_incremental)

CustomerID,CustomerName,Email,Country,UpdatedDate
102,Pavan Kumar,pavan.new@gmail.com,India,2026-08-10
106,Michael Brown,michael@gmail.com,USA,2026-08-10
107,James Wilson,james@gmail.com,Canada,2026-08-11


## Incremental MERGE 

Existing CustomerID 102 is updated; 106 and 107 are inserted.

In [ ]:
silver_table = DeltaTable.forName(
    spark,
    "silver_customers"
)

(
    silver_table.alias("target")
    .merge(
        df_incremental.alias("source"),
        "target.CustomerID = source.CustomerID"
    )
    .whenMatchedUpdate(set={
        "CustomerName": "source.CustomerName",
        "Email": "source.Email",
        "Country": "source.Country",
        "UpdatedDate": "source.UpdatedDate",
        "ingestion_timestamp": "current_timestamp()"
    })
    .whenNotMatchedInsert(values={
        "CustomerID": "source.CustomerID",
        "CustomerName": "source.CustomerName",
        "Email": "source.Email",
        "Country": "source.Country",
        "UpdatedDate": "source.UpdatedDate",
        "ingestion_timestamp": "current_timestamp()"
    })
    .execute()
)

display(spark.table("silver_customers").orderBy("CustomerID"))

CustomerID,CustomerName,Email,Country,UpdatedDate,ingestion_timestamp
101,John Smith,john@gmail.com,USA,2026-08-01,2026-08-19T06:49:16.114Z
102,Pavan Kumar,pavan.new@gmail.com,India,2026-08-10,2026-08-19T07:00:06.670Z
103,David Lee,david@gmail.com,UK,2026-08-02,2026-08-19T06:49:16.114Z
104,Sarah Jones,sarah@gmail.com,USA,2026-08-02,2026-08-19T06:49:16.114Z
105,Ravi Kumar,ravi@gmail.com,India,2026-08-03,2026-08-19T06:49:16.114Z
106,Michael Brown,michael@gmail.com,USA,2026-08-10,2026-08-19T07:00:06.670Z
107,James Wilson,james@gmail.com,Canada,2026-08-11,2026-08-19T07:00:06.670Z


## Gold - customer summary

In [ ]:
df_gold = (
    spark.table("silver_customers")
    .groupBy("Country")
    .agg(F.count("*").alias("CustomerCount"))
    .orderBy(F.desc("CustomerCount"))
)

df_gold.write.format("delta").mode("overwrite").saveAsTable("gold_customer_summary")

display(spark.table("gold_customer_summary"))

Country,CustomerCount
USA,3
India,2
Canada,1
UK,1


## validation

In [ ]:
final_df = spark.table("silver_customers")

print("Final customer count:", final_df.count())

display(
    final_df.select(
        "CustomerID", "CustomerName", "Email", "Country", "UpdatedDate"
    ).orderBy("CustomerID")
)


Final customer count: 7


CustomerID,CustomerName,Email,Country,UpdatedDate
101,John Smith,john@gmail.com,USA,2026-08-01
102,Pavan Kumar,pavan.new@gmail.com,India,2026-08-10
103,David Lee,david@gmail.com,UK,2026-08-02
104,Sarah Jones,sarah@gmail.com,USA,2026-08-02
105,Ravi Kumar,ravi@gmail.com,India,2026-08-03
106,Michael Brown,michael@gmail.com,USA,2026-08-10
107,James Wilson,james@gmail.com,Canada,2026-08-11


In [ ]:
df_final = spark.table("silver_customers")

print("Total Customers:", df_final.count())

print(
    "Null Customer IDs:",
    df_final.filter(F.col("CustomerID").isNull()).count()
)

print(
    "Null Emails:",
    df_final.filter(F.col("Email").isNull()).count()
)

print(
    "Duplicate Customer IDs:",
    df_final.groupBy("CustomerID")
            .count()
            .filter(F.col("count") > 1)
            .count()
)

Total Customers: 7
Null Customer IDs: 0
Null Emails: 0
Duplicate Customer IDs: 0


## Performance optimization



In [ ]:
# Optional for larger tables:
# spark.sql("OPTIMIZE silver_customers")

# Inspecting the Delta table
spark.sql("""
DESCRIBE DETAIL silver_customers
""").show(truncate=False)

+------+------------------------------------+----------------------------------+-----------+--------+-----------------------+-------------------+----------------+-----------------+--------+-----------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------+----------------+-----------------------------------------+---------------------------------------------------------------+-------------+
|format|id                                  |name                              |description|location|createdAt              |lastModified       |partitionColumns|clusteringColumns|numFiles|sizeInBytes|properties                                                                                                                                                                 |minReaderVersion|minWriterVersion|tableFeatures                            |statistics             

**** FINAL Validation `Summary`**

In [ ]:
final_df = spark.table("silver_customers")

print("===== POC VALIDATION =====")
print(f"Final customer count : {final_df.count()}")

updated_customer = (
    final_df
    .filter(F.col("CustomerID") == 102)
    .select("CustomerID", "Email", "UpdatedDate")
)

print("Customer 102 - Updated record:")
display(updated_customer)

print("New customers:")
display(
    final_df
    .filter(F.col("CustomerID").isin(106, 107))
    .orderBy("CustomerID")
)

===== POC VALIDATION =====
Final customer count : 7
Customer 102 - Updated record:


CustomerID,Email,UpdatedDate
102,pavan.new@gmail.com,2026-08-10


New customers:


CustomerID,CustomerName,Email,Country,UpdatedDate,ingestion_timestamp
106,Michael Brown,michael@gmail.com,USA,2026-08-10,2026-08-19T07:00:06.670Z
107,James Wilson,james@gmail.com,Canada,2026-08-11,2026-08-19T07:00:06.670Z
